In [5]:
import json
import os
from tqdm import tqdm
from collections import defaultdict

In [6]:
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)
def load_embedding_cache(path):
   
    with open(path, "w", encoding="utf-8") as f:
        json.dump({}, f)
    return {}

In [7]:
METADATA_PATH = r"data\metadata_medhop.json"
metadata = load_json(METADATA_PATH)
mapping_data = load_json(r'data\mapping_answer_gol_add_seq.json')

In [8]:
def count_token(text):
    """
    Đếm số lượng token bằng cách split khoảng trắng.
    """
    if not text or not isinstance(text, str):
        return 0
    return len(text.split())

In [9]:
dict_doc_ctx = defaultdict(int)
dict_crawl_ctx = defaultdict(int)
dict_total = defaultdict(int)


# 1000

In [10]:
BASE_1000 = "data/ALL_ROUND/1000/medhop_results_gg_direct/all_ans"
BASE_SEQ = "SEQ_medhop_results/all_ans"
BASE_PAR = "data/ALL_ROUND/1000/medhop_results_ct6_r36_parallel_seq/all_ans"

In [ ]:
for qidx in tqdm(list(metadata.keys())):
    item = metadata[qidx]
    t_q = item.get('type_q')
    
    # Xác định đường dẫn file tương ứng
    if t_q == 'direct':
        file_path = f"{BASE_1000}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'sequence':
        file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'parallel':
        file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    else:
        continue

    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue

    nodes_to_process = []

    if t_q == 'direct':
        step_data = tmp_data.get('1', {}).get('0', {}).get('0', {})
        nodes_to_process.append((f"{qidx}|1|0|0", step_data))

    elif t_q == 'sequence':
        for step, step_data in tmp_data.get('1', {}).get('0', {}).items():
            nodes_to_process.append((f"{qidx}|1|0|{step}", step_data))

    elif t_q == 'parallel':
        for qi in ["1", "2", "3", "4"]:
            if qi not in tmp_data: 
                break
            for step, step_data in tmp_data[qi].get('0', {}).items():
                nodes_to_process.append((f"{qidx}|{qi}|0|{step}", step_data))
                
    for ID, s_data in nodes_to_process:
        cluster_docs_ctx = s_data.get('retrieval_information', {})
        if not cluster_docs_ctx:
            continue
            
        for cluster_label, sample_cluster in cluster_docs_ctx.items():
            ID_ctx = f"{ID}|{cluster_label}"
            # Gom text từ list docs và crawl_doc
            docs_text = " ".join(sample_cluster.get('docs', []))
            crawl_text = " ".join(sample_cluster.get('crawl_doc', []))
            
            # Tính toán token
            d_tokens = count_token(docs_text)
            c_tokens = count_token(crawl_text) 
            
            dict_doc_ctx[ID_ctx] += d_tokens
            dict_crawl_ctx[ID_ctx] += c_tokens
            dict_total[ID_ctx] += (d_tokens + c_tokens)

#

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:07<00:00, 135.03it/s]


# 9000

In [10]:
BASE_9000 = "DIRECT_9000_medhop_results/all_ans"
BASE_SEQ = "SEQUENCE_9000_medhop_results/all_ans"
BASE_PAR = "PARALLEL_9000_medhop_results/all_ans"
metadata_9000 = load_json(r'data\metadata_9000q.json')

In [11]:
for qidx in tqdm(list(metadata_9000.keys())):
    item = metadata_9000[qidx]
    t_q = item.get('type_q')
    
    # Xác định đường dẫn file tương ứng
    if t_q == 'direct':
        file_path = f"{BASE_9000}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'sequence':
        file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'parallel':
        file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    else:
        continue

    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue

    nodes_to_process = []

    if t_q == 'direct':
        step_data = tmp_data.get('1', {}).get('0', {}).get('0', {})
        nodes_to_process.append((f"{qidx}|1|0|0", step_data))

    elif t_q == 'sequence':
        for step, step_data in tmp_data.get('1', {}).get('0', {}).items():
            nodes_to_process.append((f"{qidx}|1|0|{step}", step_data))

    elif t_q == 'parallel':
        for qi in ["1", "2", "3", "4"]:
            if qi not in tmp_data: 
                break
            for step, step_data in tmp_data[qi].get('0', {}).items():
                nodes_to_process.append((f"{qidx}|{qi}|0|{step}", step_data))
                
    for ID, s_data in nodes_to_process:
        cluster_docs_ctx = s_data.get('retrieval_information', {})
        if not cluster_docs_ctx:
            continue
            
        for cluster_label, sample_cluster in cluster_docs_ctx.items():
            # Gom text từ list docs và crawl_doc
            docs_text = " ".join(sample_cluster.get('docs', []))
            crawl_text = " ".join(sample_cluster.get('crawl_doc', []))
            
            # Tính toán token
            d_tokens = count_token(docs_text)
            c_tokens = count_token(crawl_text) 
            
            dict_doc_ctx[ID] += d_tokens
            dict_crawl_ctx[ID] += c_tokens
            dict_total[ID] += (d_tokens + c_tokens)

100%|██████████| 9000/9000 [01:08<00:00, 131.48it/s]


In [12]:
len(dict_total.keys())

12252

In [13]:
save_json(dict_doc_ctx, r'data\EVIDENCE\analysis\dict_doc_ctx_row4.json')
save_json(dict_crawl_ctx, r'data\EVIDENCE\analysis\dict_crawl_ctx_row4.json')
save_json(dict_total, r'data\EVIDENCE\analysis\dict_total_row4.json')

# conext tung cluster

In [12]:
dict_ctx_for_cluster = defaultdict(int)

BASE_1000 = "data/ALL_ROUND/1000/medhop_results_gg_direct/all_ans"
BASE_SEQ = "SEQ_medhop_results/all_ans"
BASE_PAR = "data/ALL_ROUND/1000/medhop_results_ct6_r36_parallel_seq/all_ans"

In [13]:
for qidx in tqdm(list(metadata.keys())):
    item = metadata[qidx]
    t_q = item.get('type_q')
    
    # Xác định đường dẫn file tương ứng
    if t_q == 'direct':
        file_path = f"{BASE_1000}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'sequence':
        file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'parallel':
        file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    else:
        continue

    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue

    nodes_to_process = []

    if t_q == 'direct':
        step_data = tmp_data.get('1', {}).get('0', {}).get('0', {})
        nodes_to_process.append((f"{qidx}|1|0|0", step_data))

    elif t_q == 'sequence':
        for step, step_data in tmp_data.get('1', {}).get('0', {}).items():
            nodes_to_process.append((f"{qidx}|1|0|{step}", step_data))

    elif t_q == 'parallel':
        for qi in ["1", "2", "3", "4"]:
            if qi not in tmp_data: 
                break
            for step, step_data in tmp_data[qi].get('0', {}).items():
                nodes_to_process.append((f"{qidx}|{qi}|0|{step}", step_data))
                
    for ID, s_data in nodes_to_process:
        cluster_docs_ctx = s_data.get('cluster_genered_answer', {})
        if not cluster_docs_ctx:
            continue
            
        for cluster_label, sample_cluster in cluster_docs_ctx.items():
            ID_ctx = f"{ID}|{cluster_label}"
            
            dict_ctx_for_cluster[ID_ctx] += count_token(sample_cluster['context'])

100%|██████████| 1000/1000 [00:24<00:00, 40.29it/s]


In [ ]:
BASE_9000 = "DIRECT_9000_medhop_results/all_ans"
BASE_SEQ = "SEQUENCE_9000_medhop_results/all_ans"
BASE_PAR = "PARALLEL_9000_medhop_results/all_ans"

In [15]:
for qidx in tqdm(list(metadata_9000.keys())):
    item = metadata_9000[qidx]
    t_q = item.get('type_q')
    
    # Xác định đường dẫn file tương ứng
    if t_q == 'direct':
        file_path = f"{BASE_9000}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'sequence':
        file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'parallel':
        file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    else:
        continue


    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue

    nodes_to_process = []

    if t_q == 'direct':
        step_data = tmp_data.get('1', {}).get('0', {}).get('0', {})
        nodes_to_process.append((f"{qidx}|1|0|0", step_data))

    elif t_q == 'sequence':
        for step, step_data in tmp_data.get('1', {}).get('0', {}).items():
            nodes_to_process.append((f"{qidx}|1|0|{step}", step_data))

    elif t_q == 'parallel':
        for qi in ["1", "2", "3", "4"]:
            if qi not in tmp_data: 
                break
            for step, step_data in tmp_data[qi].get('0', {}).items():
                nodes_to_process.append((f"{qidx}|{qi}|0|{step}", step_data))
                
    for ID, s_data in nodes_to_process:
        cluster_docs_ctx = s_data.get('cluster_genered_answer', {})
        if not cluster_docs_ctx:
            continue
            
        for cluster_label, sample_cluster in cluster_docs_ctx.items():
            ID_ctx = f"{ID}|{cluster_label}"
            
            dict_ctx_for_cluster[ID_ctx] += count_token(sample_cluster['context'])

100%|██████████| 9000/9000 [04:33<00:00, 32.88it/s]


In [16]:
save_json(dict_ctx_for_cluster, r'data\EVIDENCE\analysis\dict_ctx_for_cluster_row_4.json')

# ctx long answer

In [20]:
dict_long_ans_ctx = defaultdict(int)

for qidx in tqdm(list(metadata.keys())):
    item = metadata[qidx]
    t_q = item.get('type_q')
    
    # Xác định đường dẫn file tương ứng
    if t_q == 'direct':
        file_path = f"{BASE_1000}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'sequence':
        file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'parallel':
        file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    else:
        continue

    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue

    nodes_to_process = []

    if t_q == 'direct':
        step_data = tmp_data.get('1', {}).get('0', {}).get('0', {})
        nodes_to_process.append((f"{qidx}|1|0|0", step_data))

    elif t_q == 'sequence':
        for step, step_data in tmp_data.get('1', {}).get('0', {}).items():
            nodes_to_process.append((f"{qidx}|1|0|{step}", step_data))

    elif t_q == 'parallel':
        for qi in ["1", "2", "3", "4"]:
            if qi not in tmp_data: 
                break
            for step, step_data in tmp_data[qi].get('0', {}).items():
                nodes_to_process.append((f"{qidx}|{qi}|0|{step}", step_data))
                
    for ID, s_data in nodes_to_process:
        cluster_docs_ctx = s_data.get('cluster_genered_answer', {})
        if not cluster_docs_ctx:
            continue
            
        for cluster_label, sample_cluster in cluster_docs_ctx.items():
            # Gom text từ list docs và crawl_doc
            long_answer = sample_cluster['llm']['long_answer']            
            # Tính toán token
            
            dict_long_ans_ctx[ID] += count_token(long_answer)

100%|██████████| 1000/1000 [00:02<00:00, 336.18it/s]


In [21]:
len(dict_long_ans_ctx.keys())

704

In [18]:
for qidx in tqdm(list(metadata_9000.keys())):
    item = metadata_9000[qidx]
    t_q = item.get('type_q')
    
    # Xác định đường dẫn file tương ứng
    if t_q == 'direct':
        file_path = f"{BASE_9000}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'sequence':
        file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'parallel':
        file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    else:
        continue

    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue

    nodes_to_process = []

    if t_q == 'direct':
        step_data = tmp_data.get('1', {}).get('0', {}).get('0', {})
        nodes_to_process.append((f"{qidx}|1|0|0", step_data))

    elif t_q == 'sequence':
        for step, step_data in tmp_data.get('1', {}).get('0', {}).items():
            nodes_to_process.append((f"{qidx}|1|0|{step}", step_data))

    elif t_q == 'parallel':
        for qi in ["1", "2", "3", "4"]:
            if qi not in tmp_data: 
                break
            for step, step_data in tmp_data[qi].get('0', {}).items():
                nodes_to_process.append((f"{qidx}|{qi}|0|{step}", step_data))
                
    for ID, s_data in nodes_to_process:
        cluster_docs_ctx = s_data.get('cluster_genered_answer', {})
        if not cluster_docs_ctx:
            continue
            
        for cluster_label, sample_cluster in cluster_docs_ctx.items():
            # Gom text từ list docs và crawl_doc
            long_answer = sample_cluster['llm']['long_answer']            
            # Tính toán token
            
            dict_long_ans_ctx[ID] += count_token(long_answer)

100%|██████████| 9000/9000 [00:44<00:00, 202.63it/s]


In [19]:
len(dict_long_ans_ctx.keys())

11594

In [22]:
save_json(dict_long_ans_ctx, r'data\EVIDENCE\analysis\dict_long_ans_ctx_row_7.json')

# zeroshot original question

In [4]:
total_results = {}

In [ ]:
data1 = load_json(r'experiment_o3_ZEROSHOT\direct_original_question_1000.json')

for qidx in tqdm(list(data1.keys())):
    if qidx not in total_results:
        total_results.append()